In [1]:
# Put import statements here
import os
# hide tensorflow info/warning logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import sys
import subprocess
from pathlib import Path


# Local files/code
import src.data_preprocessing.image_preprocessing as img_pre
import src.data_preprocessing.text_preprocessing as text_pre
import src.data_preprocessing.text_data_exploration as text_explore
from src.agents.Visual_Agent import VisualModel
from src.util.logger import Logger
from src.util.config import config, PROJECT_ROOT
import src.util.general as general_util
from src.data_preprocessing.TextTokenizer import TextTokenizer


import sys
import tensorflow as tf
print(sys.executable)
print(tf.__version__)

I0000 00:00:1789187402.666928    6979 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789187404.342421    6979 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


/home/chris/capstone/PlantQA/venv/bin/python3.10
2.21.0


In [2]:
# Run pytest setup tests to verify the environment/hardware is good to go
#subprocess.run(
#    [sys.executable, "-m", "pytest", "-q", "test/test_setup.py"],
#    check=False,
#)

In [3]:
# Determine the project's data repo location
DATA_DIR= PROJECT_ROOT / "data"


In [4]:
# load the plantExpertVQA training, validation, and testing datasets
plant_expert_vqa_data_path= DATA_DIR / config.data.PlantExpertVQA.data_path
plant_expert_vqa_path= DATA_DIR / "PlantExpertVQA"
train_data_path= plant_expert_vqa_data_path / config.data.PlantExpertVQA.train_file
test_data_path= plant_expert_vqa_data_path / config.data.PlantExpertVQA.test_file
validation_data_path= plant_expert_vqa_data_path / config.data.PlantExpertVQA.validation_file


plant_expert_vqa_TRAIN= text_pre.load_csv(train_data_path)
plant_expert_vqa_TEST= text_pre.load_csv(test_data_path)
plant_expert_vqa_VAL= text_pre.load_csv(validation_data_path)

21:30:08 [DEBUG   ]: [load_csv] Successfully loaded /home/chris/capstone/PlantQA/data/PlantExpertVQA/data/train.csv into dataframe


/home/chris/capstone/PlantQA/src/data_preprocessing/text_preprocessing.py:27: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  data= pd.read_csv(csv_path)


21:30:08 [DEBUG   ]: [load_csv] Successfully loaded /home/chris/capstone/PlantQA/data/PlantExpertVQA/data/test.csv into dataframe
21:30:09 [DEBUG   ]: [load_csv] Successfully loaded /home/chris/capstone/PlantQA/data/PlantExpertVQA/data/val.csv into dataframe


In [5]:
PEVQA_text_columns= general_util.parse_list_from_string(config.data.PlantExpertVQA.text_columns)
PEVQA_columns_to_remove= general_util.parse_list_from_string(config.data.PlantExpertVQA.columns_to_remove)
PEVQA_na_fill= vars(config.data.PlantExpertVQA.na_fill)

# preprocess (not tokenize) the training, testing, and validation datasets
# NOTE: the stop word removal may be too intense.  We most likely want to fine tune the stopword set, or define our own set
#       Right now it removes words like "what" and "why", which will most likely be bad for a VQA system that answers questions.

# training
text_pre.preprocess_dataframe(
    plant_expert_vqa_TRAIN,
    PEVQA_text_columns,
    PEVQA_columns_to_remove,
    PEVQA_na_fill,
    ["image_path"],
    plant_expert_vqa_path,
)

# testing
text_pre.preprocess_dataframe(
    plant_expert_vqa_TEST,
    PEVQA_text_columns,
    PEVQA_columns_to_remove,
    PEVQA_na_fill,
    ["image_path"],
    plant_expert_vqa_path,
)

#validation
text_pre.preprocess_dataframe(
    plant_expert_vqa_VAL,
    PEVQA_text_columns,
    PEVQA_columns_to_remove,
    PEVQA_na_fill,
    ["image_path"],
    plant_expert_vqa_path,
)


In [6]:

column_distribution_args= [
    {"column": "crop", "show_counts": False, "figure_size": (10, 5)},
    {"column": "severity", "show_counts": True, "figure_size": (5, 5)},
    {"column": "category", "show_counts": True, "figure_size": (5, 5)},
    {"column": "answer_type", "show_counts": True, "figure_size": (5, 5)},
    {"column": "question_category", "show_counts": False, "figure_size": (10, 5)},
]

# explore the cleaned training dataset
"""text_explore.explore_data(
    plant_expert_vqa_TRAIN, 
    column_distribution_args, 
    PEVQA_text_columns, 
    top_n_words=20, 
    name="Plant Expert VQA Training Dataset"
)"""

'text_explore.explore_data(\n    plant_expert_vqa_TRAIN, \n    column_distribution_args, \n    PEVQA_text_columns, \n    top_n_words=20, \n    name="Plant Expert VQA Training Dataset"\n)'

In [7]:
tokenizer= TextTokenizer("distilbert-base-uncased", 128)

questions= plant_expert_vqa_TRAIN["question_text"]
print(plant_expert_vqa_TRAIN.columns)

inputs, attention_mask= tokenizer.encode_text(questions)

Logger.info(inputs)
Logger.info(type(inputs))

Logger.info(attention_mask)
Logger.info(type(attention_mask))

Index(['qa_id', 'image_id', 'image_path', 'crop', 'disease', 'category',
       'severity', 'question_text', 'answer', 'answer_type',
       'question_category', 'cognitive_level'],
      dtype='object')
21:24:37 [INFO    ]: tf.Tensor(
[[  101  4295  3491 ...     0     0     0]
 [  101  5729  2175 ...     0     0     0]
 [  101  2419  5729 ...     0     0     0]
 ...
 [  101 20856  7053 ...     0     0     0]
 [  101 11487  2004 ...     0     0     0]
 [  101 20856  7053 ...     0     0     0]], shape=(535881, 20), dtype=int64)
21:24:37 [INFO    ]: <class 'tensorflow.python.framework.ops.EagerTensor'>
21:24:37 [INFO    ]: tf.Tensor(
[[1 1 1 ... 0 0 0]
 [1 1 1 ... 0 0 0]
 [1 1 1 ... 0 0 0]
 ...
 [1 1 1 ... 0 0 0]
 [1 1 1 ... 0 0 0]
 [1 1 1 ... 0 0 0]], shape=(535881, 20), dtype=int64)
21:24:37 [INFO    ]: <class 'tensorflow.python.framework.ops.EagerTensor'>


In [8]:
# Adding on macbook - will need to retest on workstation later tonight due to GPU acceleration
IMAGES_ROOT = DATA_DIR / "PlantExpertVQA"
traits = ["crop", "disease", "category", "severity"]

classes = {}
for t in traits:
    names = sorted(plant_expert_vqa_TRAIN[t].unique())  # adds names of plants
    classes[t] = names + ["unknown"]  # adds unknown species

train_ds = VisualModel.make_dataset(plant_expert_vqa_TRAIN, classes, IMAGES_ROOT, training=True)
val_ds = VisualModel.make_dataset(plant_expert_vqa_VAL, classes, IMAGES_ROOT)

visual_model = VisualModel(classes=classes)
visual_model.build()
visual_model.compile()
visual_model.fit(train_ds, val_ds, epochs=20)

Epoch 1/20
   39/16747 ━━━━━━━━━━━━━━━━━━━━ 45:26:10 10s/step - category_accuracy: 0.9140 - category_loss: 0.4826 - crop_accuracy: 0.7250 - crop_loss: 1.7251 - disease_accuracy: 0.5010 - disease_loss: 2.6343 - loss: 4.7179 - severity_accuracy: 0.8046 - severity_loss: 0.7126 

KeyboardInterrupt: 

In [ ]:
import json
Path("./models/visual_models/9_10_classes.json").write_text(json.dumps(classes))
visual_model.save(path="./models/visual_models/9_10.keras")